# Xiaomi Price Checker

Notebook controller for reading the Excel configuration, running the enabled website scrapers, collecting variant-level price/availability data, and writing the results to `RawData`.

The actual website scraping remains in `scrapers/` and is executed through `run_price_check.py`.


In [ ]:
# ============================================================
# 1. SETUP
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys

from openpyxl import load_workbook
import pandas as pd

project_folder = Path(
    r"C:\Users\garli\Python_Projects\XM_Price_Checker"
)

excel_file = project_folder / "XM_Price_Checker_Python.xlsm"

print("Project folder:", project_folder)
print("Excel exists:", excel_file.exists())


: 

In [72]:
# ============================================================
# 2. LOAD EXCEL WORKBOOK
# ============================================================

workbook = load_workbook(
    excel_file,
    keep_vba=True
)

products_sheet = workbook["Products"]
links_sheet = workbook["Links"]
rawdata_sheet = workbook["RawData"]
results_sheet = workbook["Results"]

print("Workbook loaded successfully.")
print("Sheets:", workbook.sheetnames)
print("Products rows:", products_sheet.max_row)
print("Links rows:", links_sheet.max_row)
print("RawData rows:", rawdata_sheet.max_row)
print("Results rows:", results_sheet.max_row)


Workbook loaded successfully.
Sheets: ['Products', 'Links', 'RawData', 'Results', 'Settings']
Products rows: 31
Links rows: 57
RawData rows: 1


In [73]:
# ============================================================
# 3. LOAD PRODUCTS
# ============================================================

products = {}

for row in products_sheet.iter_rows(
    min_row=2,
    values_only=True
):
    if not row:
        continue

    model = row[1]
    name = row[2]
    ram = row[3]
    storage = row[4]

    if model is None or ram is None or storage is None:
        continue

    model = str(model).strip()
    ram = str(ram).strip()
    storage = str(storage).strip()

    target_id = f"{model}-{ram}-{storage}"

    products[target_id.upper()] = {
        "target_id": target_id,
        "name": str(name).strip() if name is not None else model,
        "model": model,
        "ram": ram,
        "storage": storage
    }

print("Products loaded:", len(products))

if products:
    print("Example:", next(iter(products.values())))


Products loaded: 29
Example: {'target_id': 'SOMALIAA-4-64', 'name': 'Redmi A7 pro', 'model': 'SOMALIAA', 'ram': '4', 'storage': '64'}


In [74]:
# ============================================================
# 4. LOAD ENABLED LINKS
# ============================================================

links = []

for row in links_sheet.iter_rows(
    min_row=2,
    values_only=True
):
    if not row:
        continue

    target_id = row[0]
    website = row[1]
    url = row[2]
    enabled = row[3]

    if (
        target_id
        and website
        and url
        and str(enabled).strip().upper() == "YES"
    ):
        links.append({
            "target_id": str(target_id).strip(),
            "website": str(website).strip().upper(),
            "url": str(url).strip()
        })

target_ids = sorted({
    item["target_id"].strip().upper()
    for item in links
})

print("Enabled links:", len(links))
print("Target IDs:", target_ids)


Enabled links: 32
Target IDs: ['O19AE-6-128', 'O19AE-8-256', 'P15AE-4-128', 'SOMALIAA-4-64']


In [75]:
# ============================================================
# 5. RUN PRICE CHECKER
# ============================================================

def run_price_check(website, url, product):
    """Run the website scraper in a separate Python process."""

    script = project_folder / "run_price_check.py"

    product_json = json.dumps(
        product,
        ensure_ascii=False
    )

    process = subprocess.run(
        [
            sys.executable,
            str(script),
            website,
            url,
            product_json
        ],
        capture_output=True,
        text=True,
        encoding="utf-8"
    )

    # Keep scraper/browser logs visible in the notebook.
    print(process.stdout)

    if process.returncode != 0:
        print(process.stderr)
        raise RuntimeError(
            f"{website} price check failed."
        )

    for line in process.stdout.splitlines():
        if line.startswith("RESULT_JSON:"):
            return json.loads(
                line.replace(
                    "RESULT_JSON:",
                    "",
                    1
                ).strip()
            )

    raise ValueError(
        f"No RESULT_JSON returned for {website}."
    )


In [76]:
# ============================================================
# 6. PRODUCT LOOKUP
# ============================================================

def get_product(target_id):
    """Return product information loaded from the Products sheet."""

    key = str(target_id).strip().upper()

    try:
        return products[key]
    except KeyError:
        raise ValueError(
            f"TargetID not found in Products: {target_id}"
        )


In [77]:
# ============================================================
# 7. PRICE COLLECTION
# ============================================================

all_results = []

for target_id in target_ids:

    product = get_product(target_id)

    print()
    print("=" * 60)
    print("TARGET:", target_id)
    print(
        "Product:",
        product["name"],
        "| RAM:", product["ram"],
        "| Storage:", product["storage"]
    )
    print("=" * 60)

    target_links = [
        item for item in links
        if item["target_id"].strip().upper() == target_id
    ]

    for item in target_links:

        website = item["website"]
        url = item["url"]

        print()
        print("-" * 50)
        print("Website:", website)
        print("URL:", url)

        try:
            results = run_price_check(
                website,
                url,
                product
            )

            print("SCRAPER RESULTS:", results)

            # No matching product is not a scraper error.
            if not results:
                print("No matching products found.")
                continue

            for result in results:
                all_results.append({
                    "run_time": datetime.now(),
                    "target_id": target_id,
                    "website": website,
                    "product_name": result.get(
                        "product_name",
                        product["name"]
                    ),
                    "variant": result.get("variant", ""),
                    "ram": result.get("ram", product["ram"]),
                    "storage": result.get("storage", product["storage"]),
                    "url": url,
                    "price": result.get("price"),
                    "currency": "PLN",
                    "availability": result.get(
                        "availability",
                        "Unknown"
                    ),
                    "status": "OK",
                    "error": ""
                })

        except Exception as e:
            print("ERROR:", str(e))

            all_results.append({
                "run_time": datetime.now(),
                "target_id": target_id,
                "website": website,
                "product_name": product["name"],
                "variant": "",
                "ram": product["ram"],
                "storage": product["storage"],
                "url": url,
                "price": None,
                "currency": "PLN",
                "availability": "Unknown",
                "status": "ERROR",
                "error": str(e)
            })

print()
print("=" * 60)
print("PRICE COLLECTION COMPLETE")
print("=" * 60)
print("Total results:", len(all_results))



TARGET: O19AE-6-128
Product: Redmi 15 | RAM: 6 | Storage: 128

--------------------------------------------------
Website: MEX
URL: https://www.mediaexpert.pl/search?query[menu_item]=&query[querystring]=Redmi%2015%206%2F128
AVANS MODULE PATH: C:\Users\New\.vscode\Python_projects\XM_Price_Checker\scrapers\avans.py
NEONET MODULE PATH: C:\Users\New\.vscode\Python_projects\XM_Price_Checker\scrapers\neonet.py
Website: MEX
URL: https://www.mediaexpert.pl/search?query[menu_item]=&query[querystring]=Redmi%2015%206%2F128
Product: {'target_id': 'O19AE-6-128', 'name': 'Redmi 15', 'model': 'O19AE', 'ram': '6', 'storage': '128'}
Opening: https://www.mediaexpert.pl/search?query[menu_item]=&query[querystring]=Redmi%2015%206%2F128
Page loaded.
Loading all Media Expert products...
Current h2/h3 count: 48
Current h2/h3 count: 50
Current h2/h3 count: 58
Current h2/h3 count: 63
Current h2/h3 count: 59
Current h2/h3 count: 59
Current h2/h3 count: 59
Finished loading products. Total h2/h3: 59
Total h2/h3 e

In [78]:
# ============================================================
# 8. RESULTS / VERIFICATION
# ============================================================

results_df = pd.DataFrame(all_results)

print("Raw scraper results:", len(results_df), "rows")

if not results_df.empty:
    display(
        results_df[
            [
                "target_id",
                "website",
                "product_name",
                "variant",
                "ram",
                "storage",
                "price",
                "availability",
                "status",
                "error"
            ]
        ]
    )

    print()
    print("=" * 70)
    print("TARGET / WEBSITE SUMMARY")
    print("=" * 70)

    summary_df = (
        results_df
        .groupby(
            ["target_id", "website"],
            dropna=False
        )
        .agg(
            variants=("variant", "count"),
            available=(
                "availability",
                lambda x: (x == "Available").sum()
            ),
            unavailable=(
                "availability",
                lambda x: (x == "Unavailable").sum()
            ),
            prices=(
                "price",
                lambda x: x.notna().sum()
            ),
            errors=(
                "status",
                lambda x: (x == "ERROR").sum()
            )
        )
        .reset_index()
    )

    display(summary_df)
else:
    print("No scraper results were collected.")


Results: 76 rows


,target_id,website,product_name,variant,ram,storage,price,availability,status,error
0,O19AE-6-128,MEX,Redmi 15,Black,6,128,649.0,Available,OK,
1,O19AE-6-128,MEX,Redmi 15,Purple,6,128,NaN,Unavailable,OK,
2,O19AE-6-128,MEX,Redmi 15,Titanium,6,128,NaN,Unavailable,OK,
3,O19AE-6-128,ELECTRO.PL,Redmi 15,Black,6 GB,128 GB,NaN,Unavailable,OK,
4,O19AE-6-128,ELECTRO.PL,Redmi 15,Purple,6 GB,128 GB,NaN,Unavailable,OK,
...,...,...,...,...,...,...,...,...,...,...
71,SOMALIAA-4-64,NEONET,Redmi A7 pro,Black,4 GB,64 GB,449.0,Available,OK,
72,SOMALIAA-4-64,NEONET,Redmi A7 pro,Blue,4 GB,64 GB,NaN,Available,OK,
73,SOMALIAA-4-64,MAX ELECTRO,Redmi A7 pro,Blue,4 GB,64 GB,399.0,Available,OK,
74,SOMALIAA-4-64,MAX ELECTRO,Redmi A7 pro,Black,4 GB,64 GB,399.0,Available,OK,



TARGET / WEBSITE SUMMARY


,target_id,website,variants,available,unavailable,prices,errors
0,O19AE-6-128,AVANS,3,0,3,0,0
1,O19AE-6-128,ELECTRO.PL,3,0,3,0,0
2,O19AE-6-128,KTR,1,1,0,1,0
3,O19AE-6-128,MAX ELECTRO,1,0,1,0,0
4,O19AE-6-128,MEX,3,1,2,1,0
5,O19AE-6-128,MSH,3,3,0,3,0
6,O19AE-6-128,NEONET,1,0,1,0,0
7,O19AE-6-128,XKOM,1,0,1,0,0
8,O19AE-8-256,AVANS,3,2,1,2,0
9,O19AE-8-256,ELECTRO.PL,3,2,1,2,0


In [79]:
# ============================================================
# 9. RAW DATA OUTPUT
# ============================================================
#
# The RawData sheet is append-only.
# If you want a fresh run instead of keeping historical rows,
# clear the existing RawData rows manually before running this cell.
# ============================================================

for _, result in results_df.iterrows():

    price = result["price"]

    # Pandas NaN -> blank Excel cell
    if pd.isna(price):
        price = None

    rawdata_sheet.append([
        result["run_time"],
        result["target_id"],
        result["website"],
        result["product_name"],
        result["variant"],
        result["url"],
        price,
        result["currency"],
        result["availability"],
        result["status"],
        result["error"]
    ])

print("RawData rows written:", len(results_df))


RawData rows written: 76


In [ ]:
# ============================================================
# 10. GENERATE RESULTS SHEET
# ============================================================
#
# The Results sheet is the final report.
#
# IMPORTANT:
# - Matching between Products and RawData uses TargetID.
# - RRP is displayed but is NOT used for the comparison.
# - Only "RRP after Promotion" is used as the benchmark.
# - Offer start/end dates are copied to the report but are NOT used.
#
# Report logic for each website:
#
# 1. If all returned variants are Unavailable -> "unavailable"
# 2. Otherwise ignore unavailable variants and compare the
#    available prices with RRP after Promotion.
# 3. If the lowest available price == promotion price -> "OK"
#    (even if another colour is more expensive).
# 4. If the lowest available price is below promotion price ->
#    "<price> <colour>".
# 5. If the lowest available price is above promotion price ->
#    "<price> <colour>" unless every available colour has the
#    same price, in which case only "<price>" is returned.
# ============================================================

def normalize_number(value):
    """Return a numeric value as float, or None."""

    if value is None or pd.isna(value):
        return None

    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def build_report_value(raw_rows, promotion_price):
    """Build one website cell for the final Results report."""

    if not raw_rows:
        # No scraper record at all. This is different from an
        # explicit Unavailable result.
        return ""

    promotion_price = normalize_number(promotion_price)

    # --------------------------------------------------------
    # Explicitly unavailable variants
    # --------------------------------------------------------

    available_rows = []
    unavailable_count = 0

    for row in raw_rows:

        availability = str(
            row.get("availability", "")
        ).strip().lower()

        price = normalize_number(
            row.get("price")
        )

        if availability == "unavailable":
            unavailable_count += 1
            continue

        if availability == "available" and price is not None:
            available_rows.append(row)

    # If every returned variant is unavailable, report exactly
    # "unavailable".
    if not available_rows and unavailable_count > 0:
        return "unavailable"

    # No usable price and no explicit unavailable result.
    # This can happen when a scraper failed or returned Unknown.
    if not available_rows:
        return ""

    # --------------------------------------------------------
    # Find the lowest available price
    # --------------------------------------------------------

    priced_rows = [
        row for row in available_rows
        if normalize_number(row.get("price")) is not None
    ]

    if not priced_rows:
        return ""

    min_price = min(
        normalize_number(row["price"])
        for row in priced_rows
    )

    # --------------------------------------------------------
    # Compare only against RRP after Promotion
    # --------------------------------------------------------

    if promotion_price is not None:
        # If any colour is exactly at the promotion price,
        # the website is OK as long as no colour is cheaper.
        if min_price == promotion_price:
            return "OK"

    # --------------------------------------------------------
    # Price is different from promotion price
    # --------------------------------------------------------

    min_rows = [
        row for row in priced_rows
        if normalize_number(row["price"]) == min_price
    ]

    # Get colours for the minimum price.
    colors = []

    for row in min_rows:

        color = str(
            row.get("variant", "")
        ).strip()

        if not color:
            color = "Unknown"

        # Avoid duplicate colour names.
        if color.lower() not in {
            existing.lower()
            for existing in colors
        }:
            colors.append(color)

    # If all available colours have exactly the same price,
    # return only the price.
    all_prices = [
        normalize_number(row["price"])
        for row in priced_rows
    ]

    if len(set(all_prices)) == 1:
        return str(int(min_price)) if min_price.is_integer() else str(min_price)

    # Otherwise return the lowest price plus the colour(s).
    color_text = ", ".join(
        color.lower()
        for color in colors
    )

    price_text = (
        str(int(min_price))
        if min_price.is_integer()
        else str(min_price)
    )

    return f"{price_text} {color_text}"


# ------------------------------------------------------------
# Read the Results header already prepared in Excel.
# This keeps website column names synchronized with the
# workbook, rather than hard-coding them in the notebook.
# ------------------------------------------------------------

results_header = [
    results_sheet.cell(1, col).value
    for col in range(
        1,
        results_sheet.max_column + 1
    )
]

website_columns = results_header[7:]

print("Results website columns:")
print(website_columns)


# ------------------------------------------------------------
# Clear old Results data but keep the existing formatting.
# ------------------------------------------------------------

for row in results_sheet.iter_rows(
    min_row=2,
    max_row=results_sheet.max_row,
    min_col=1,
    max_col=results_sheet.max_column
):
    for cell in row:
        cell.value = None


# ------------------------------------------------------------
# Create a lookup:
#
# (TargetID, Website) -> list of scraper results
# ------------------------------------------------------------

raw_lookup = {}

for _, raw_row in results_df.iterrows():

    target_id = str(
        raw_row["target_id"]
    ).strip().upper()

    website = str(
        raw_row["website"]
    ).strip().upper()

    key = (
        target_id,
        website
    )

    raw_lookup.setdefault(
        key,
        []
    ).append(
        raw_row.to_dict()
    )


# ------------------------------------------------------------
# Write one report row for every product in Products.
# ------------------------------------------------------------

result_row_number = 2

for product_row in products_sheet.iter_rows(
    min_row=2,
    values_only=True
):

    if not product_row:
        continue

    model = product_row[1]
    product_name = product_row[2]
    ram = product_row[3]
    storage = product_row[4]
    rrp = product_row[5]
    promotion_price = product_row[6]
    offer_start = product_row[7]
    offer_end = product_row[8]

    if (
        model is None
        or ram is None
        or storage is None
    ):
        continue

    model = str(model).strip()
    ram = str(ram).strip()
    storage = str(storage).strip()

    # IMPORTANT:
    # Products uses Model + RAM + Storage to construct TargetID.
    # Results separates SKU and Memory, but the matching key is
    # still the combined TargetID.
    target_id = (
        f"{model}-{ram}-{storage}"
    ).upper()

    # Results: SKU
    results_sheet.cell(
        result_row_number,
        1
    ).value = model

    # Results: Memory
    results_sheet.cell(
        result_row_number,
        2
    ).value = f"{ram}+{storage}"

    # Results: Product Name
    results_sheet.cell(
        result_row_number,
        3
    ).value = product_name

    # Results: RRP
    results_sheet.cell(
        result_row_number,
        4
    ).value = rrp

    # Results: RRP after Promotion
    results_sheet.cell(
        result_row_number,
        5
    ).value = promotion_price

    # Dates are copied only; they are not used in comparison.
    results_sheet.cell(
        result_row_number,
        6
    ).value = offer_start

    results_sheet.cell(
        result_row_number,
        7
    ).value = offer_end

    # --------------------------------------------------------
    # Website results
    # --------------------------------------------------------

    for offset, website_column in enumerate(
        website_columns,
        start=8
    ):

        if not website_column:
            continue

        website_key = str(
            website_column
        ).strip().upper()

        raw_rows = raw_lookup.get(
            (
                target_id,
                website_key
            ),
            []
        )

        report_value = build_report_value(
            raw_rows,
            promotion_price
        )

        results_sheet.cell(
            result_row_number,
            offset
        ).value = report_value

    result_row_number += 1


print()
print("=" * 70)
print("RESULTS SHEET GENERATED")
print("=" * 70)
print(
    "Report rows:",
    result_row_number - 2
)
print(
    "Website columns:",
    website_columns
)

# Preview the generated report.
report_preview = pd.DataFrame(
    results_sheet.values
)

display(report_preview)


In [80]:
# ============================================================
# 11. SAVE WORKBOOK
# ============================================================

workbook.save(excel_file)

print("RawData and Results updated successfully.")
print("Workbook saved:", excel_file)


RawData updated successfully.
Workbook saved: C:\Users\New\.vscode\Python_projects\XM_Price_Checker\XM_Price_Checker_Python.xlsm
